In [1]:
from pathlib import Path
import sys
import numpy as np
def ensure_tests_on_path(max_up=10):
    p = Path.cwd().resolve()
    for _ in range(max_up):
        if (p / 'tests').is_dir():
            sys.path.insert(0, str(p))
            return str(p)
        if p.parent == p:
            break
        p = p.parent
    # fallback: insert cwd
    sys.path.insert(0, str(Path.cwd().resolve()))
    return str(Path.cwd().resolve())

root_added = ensure_tests_on_path()
print("Added to sys.path:", root_added)


Added to sys.path: /Users/ashmi/Code/Scripts/phd/travel-crs


In [2]:
DIMENSIONS = ["relevance", "sustainability", "popularity_balance", "diversity"]

In [9]:
import json
from tests.analysis.src.quantitative import *
from tests.analysis.src.utils import load_listwise_evaluations_df, add_numeric_scores
import numpy as np
import pandas as pd

In [4]:
%load_ext autoreload
%autoreload 2

In [5]:
def compute_mapping(df, dimensions=None, include_confidence=False, threshold=0.7):
    """
    Return a copy of df with *_score values mapped to 'L1'/'L2'/NaN.
    If include_confidence is True and a `{dim}_confidence` column exists,
    rows with confidence < threshold will have the corresponding `{dim}_score`
    treated as NaN before mapping.
    """
    if dimensions is None:
        dimensions = ["relevance", "sustainability", "popularity_balance", "diversity"]

    df = df.copy()
    mapping = {1: "L1", 2: "L1", -1: "L2", -2: "L2", 0: np.nan, -3: np.nan}

    for dim in dimensions:
        score_col = f"{dim}_score"
        conf_col = f"{dim}_confidence"

        if score_col not in df.columns:
            continue

        # If confidence filtering requested and confidence column exists,
        # mask low-confidence rows by setting the score to NaN
        if include_confidence and conf_col in df.columns:
            low_conf_mask = df[conf_col] < threshold
            if low_conf_mask.any():
                df.loc[low_conf_mask, score_col] = np.nan

        # Map numeric codes to labels/NaN safely on the copy
        df[score_col] = df[score_col].map(mapping).astype(object)

    return df

def count_best_matches(df, dimensions, best_col="best_list", include_confidence=False, threshold=0.7):
    """
    Count per-dimension how many rows have `{dim}_score` == best_col.
    If include_confidence is True, apply the confidence threshold before counting.
    Returns a DataFrame with columns: dimension, score_col, matches, n_valid, pct.
    """
    df_mapped = compute_mapping(df, dimensions=dimensions, include_confidence=include_confidence, threshold=threshold)
    results = []
    for dim in dimensions:
        score_col = f"{dim}_score"
        if score_col not in df_mapped.columns:
            results.append({"dimension": dim, "score_col": score_col, "matches": 0, "n_valid": 0, "pct": np.nan, "note": "missing"})
            continue

        valid_mask = df_mapped[score_col].notna() & df_mapped[best_col].notna()
        n_valid = int(valid_mask.sum())
        if n_valid == 0:
            matches = 0
            pct = np.nan
        else:
            matches = int((df_mapped.loc[valid_mask, score_col] == df_mapped.loc[valid_mask, best_col]).sum())
            pct = round(matches / n_valid * 100.0, 2)
        results.append({"dimension": dim, "score_col": score_col, "matches": matches, "n_valid": n_valid, "pct": pct})
    return pd.DataFrame(results)


In [10]:
version = "v1"
def run_best_match_comparison(version):
    gemini_eval_path = f'../../data/conv-trs/ecir-2026/direct-reasoner/gemini_eval_{version}.json'
    gpt_eval_path = f'../../data/conv-trs/ecir-2026/direct-reasoner/gpt5_eval_{version}.json'
    deepseek_eval_path = f'../../data/conv-trs/ecir-2026/direct-reasoner/deepseek_eval_{version}.json'
    
    selected_cols = ["query_id", "judge_model", "best_list", "overall_confidence",\
                                                           "relevance_score", "relevance_confidence", \
                                                           "diversity_score", "sustainability_score", "sustainability_confidence", \
                                                           "popularity_balance_score", "popularity_balance_confidence"]
        
    df_gpt = load_listwise_evaluations_df(
        gpt_eval_path, model_name="gpt5", version=version)[selected_cols]
    df_gemini = load_listwise_evaluations_df(
        gemini_eval_path, model_name="gemini2.5pro", version=version)[selected_cols]
    df_deepseek = load_listwise_evaluations_df(deepseek_eval_path, model_name="deepseek", version=version)[selected_cols]
    
    # print(f"=============== GPT {version} best match count (without confidence) ===============")
    # print(count_best_matches(df_gpt, DIMENSIONS, include_confidence = False))

    print(f"\n\n=============== GPT {version} best match count (with confidence threshold: 0.7) ===============")
    print(count_best_matches(df_gpt, DIMENSIONS, include_confidence = True))

    # print(f"\n\n=============== Gemini {version} best match count (without confidence) ===============")
    # print(count_best_matches(df_gemini, DIMENSIONS, include_confidence = False))

    print(f"\n\n=============== Gemini {version} best match count (with confidence threshold: 0.7) ===============")
    print(count_best_matches(df_gemini, DIMENSIONS, include_confidence = True))

    
    # print(f"\n\n=============== Deepseek {version} best match count (without confidence) ===============")
    # print(count_best_matches(df_deepseek, DIMENSIONS, include_confidence = False))

    print(f"\n\n=============== Deepseek {version} best match count (with confidence threshold: 0.7) ===============")
    print(count_best_matches(df_deepseek, DIMENSIONS, include_confidence = True))

In [11]:
run_best_match_comparison("v1")

ℹ️ Loaded 101 successful evaluation entries for gpt5, version v1.
ℹ️ Loaded 101 successful evaluation entries for gemini2.5pro, version v1.
ℹ️ Loaded 101 successful evaluation entries for deepseek, version v1.


=============== GPT v1 best match count (with confidence threshold: 0.7) ===============
            dimension                 score_col  matches  n_valid     pct
0           relevance           relevance_score       55       65   84.62
1           diversity           diversity_score       41       70   58.57
2  popularity_balance  popularity_balance_score       38       43   88.37
3      sustainability      sustainability_score       23       23  100.00


=============== Gemini v1 best match count (with confidence threshold: 0.7) ===============
            dimension                 score_col  matches  n_valid    pct
0           relevance           relevance_score       71       85  83.53
1           diversity           diversity_score       46       85  54.12
2  popularity_ba

In [12]:
run_best_match_comparison("v2")

ℹ️ Loaded 101 successful evaluation entries for gpt5, version v2.
ℹ️ Loaded 101 successful evaluation entries for gemini2.5pro, version v2.
ℹ️ Loaded 101 successful evaluation entries for deepseek, version v2.


=============== GPT v2 best match count (with confidence threshold: 0.7) ===============
            dimension                 score_col  matches  n_valid    pct
0           relevance           relevance_score       53       57  92.98
1           diversity           diversity_score       45       81  55.56
2  popularity_balance  popularity_balance_score       33       39  84.62
3      sustainability      sustainability_score       27       30  90.00


=============== Gemini v2 best match count (with confidence threshold: 0.7) ===============
            dimension                 score_col  matches  n_valid    pct
0           relevance           relevance_score       61       76  80.26
1           diversity           diversity_score       64       94  68.09
2  popularity_balance